## Static zone features: SoilGrids and SRTM

This notebook builds the static (non-time-varying) features of each zone from two datasets.
**SoilGrids** (ISRIC, 250 m) is a modelled global soil map; we use clay, sand, silt, organic
carbon, pH and bulk density at three depths to describe water holding, drainage and soil
health. **SRTM** (NASA, 30 m) is a radar elevation model from 2000; we derive elevation and
slope to describe runoff and erosion risk. Each zone gets one row in the final table. Both
sources are predictions or snapshots, so they are used to compare zones with each other, not
as exact field measurements.

Functions

In [12]:
def srtm_clean(path, point, smooth=3):
    lat, lon = point
    with rasterio.open(path) as src:
        elev = src.read(1).astype(float)
        b = src.bounds
        checks = {
            "crs": str(src.crs),
            "size": (src.width, src.height),
            "point_inside": b.left <= lon <= b.right and b.bottom <= lat <= b.top,
        }
        elev[elev == src.nodata] = np.nan
        c_lat = (b.top + b.bottom) / 2
        dy = src.res[1] * 111320
        dx = src.res[0] * 111320 * np.cos(np.radians(c_lat))

    checks["share_missing"] = float(np.isnan(elev).mean())
    checks["elev_min_max"] = (float(np.nanmin(elev)), float(np.nanmax(elev)))

    if smooth:
        elev = uniform_filter(elev, size=smooth)

    dzdy, dzdx = np.gradient(elev, dy, dx)
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))

    stats = {
        "elev_mean_m": float(np.nanmean(elev)),
        "elev_std_m": float(np.nanstd(elev)),
        "slope_mean_deg": float(np.nanmean(slope)),
        "slope_p90_deg": float(np.nanpercentile(slope, 90)),
    }
    return checks, stats

### 1. Setup

In [14]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import uniform_filter
import requests, json, time
from pathlib import Path

### 2. About the SoilGrids data

**What it is.** SoilGrids (ISRIC) is a global map of soil properties predicted by
machine learning from soil samples and environmental data, at 250 m resolution.
It is **modelled data, not a measurement taken on our parcels**, and it is not
a NASA product.

**What we downloaded.** For each zone, six properties at three depths (0-5, 5-15 and
15-30 cm): clay, sand and silt content, soil organic carbon (`soc`), pH in water
(`phh2o`) and bulk density (`bdod`). We used [one point per zone / the median of
up to 9 points within about 1 km of the reference point]. Values were converted to
conventional units with the conversion factor in the API response
([units: % for clay, sand and silt; g/kg for organic carbon; pH; kg/dm³ for
bulk density: confirm in the raw JSON]).

**Why we use it.** Soil texture and organic carbon decide how much water a soil holds,
how fast it drains and how easily it erodes, so they separate zones that have the
same weather but react differently to drought. Organic carbon is also a direct
indicator of soil health, which is what the rotation options aim to improve.

**What we found for Écija.** About 34-38% clay, 25-27% sand and 36-39% silt (a clay
loam), alkaline pH (7.8-7.9), and organic carbon that falls from the top layer to
the deeper ones.

**Limitations.** Predictions are smooth averages over a 250 m cell and can differ
from a field sample. The surface organic carbon value for Écija looks high for arable
land, so we prefer the deeper layers or the 0-30 cm average. Some pixels (built-up
land, water) have no value.

In [6]:

ZONES = {
    "ecija":  (37.55537, -5.14244),
    "ubeda":  (38.00720, -3.26722),
    "guadix": (37.24278, -3.16138),
    "jerez":  (36.74074, -6.18370),
}

PROPS = ["clay", "sand", "silt", "soc", "phh2o", "bdod"]
DEPTHS = ["0-5cm", "5-15cm", "15-30cm"]
URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"

def fetch_soilgrids(lat, lon):
    params = {"lat": lat, "lon": lon, "property": PROPS,
              "depth": DEPTHS, "value": "mean"}
    r = requests.get(URL, params=params, timeout=120)
    r.raise_for_status()
    return r.json()

test = fetch_soilgrids(*ZONES["ecija"])
print(list(test.keys()))
print(json.dumps(test["properties"]["layers"][0], indent=1)[:1500])

['type', 'geometry', 'properties', 'query_time_s']
{
 "name": "bdod",
 "unit_measure": {
  "d_factor": 100,
  "mapped_units": "cg/cm\u00b3",
  "target_units": "kg/dm\u00b3",
  "uncertainty_unit": ""
 },
 "depths": [
  {
   "range": {
    "top_depth": 0,
    "bottom_depth": 5,
    "unit_depth": "cm"
   },
   "label": "0-5cm",
   "values": {
    "mean": 144
   }
  },
  {
   "range": {
    "top_depth": 5,
    "bottom_depth": 15,
    "unit_depth": "cm"
   },
   "label": "5-15cm",
   "values": {
    "mean": 156
   }
  },
  {
   "range": {
    "top_depth": 15,
    "bottom_depth": 30,
    "unit_depth": "cm"
   },
   "label": "15-30cm",
   "values": {
    "mean": 159
   }
  }
 ]
}


In [7]:
RAW = Path("../data/raw/soilgrids"); RAW.mkdir(parents=True, exist_ok=True)

for zone, (lat, lon) in ZONES.items():
    out = RAW / f"{zone}_soilgrids.json"
    if out.exists():
        continue                    
    out.write_text(json.dumps(fetch_soilgrids(lat, lon)))
    print("saved", zone)
    time.sleep(5)                    

saved ecija
saved ubeda
saved guadix
saved jerez


In [8]:
import json
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/soilgrids")
OUT = Path("../data/processed/soilgrids")
OUT.mkdir(parents=True, exist_ok=True)

for zone in ZONES:
    data = json.loads((RAW / f"{zone}_soilgrids.json").read_text())

    row = {}
    for layer in data["properties"]["layers"]:
        d_factor = layer["unit_measure"]["d_factor"]
        for depth in layer["depths"]:
            val = depth["values"].get("mean")
            row[f"{layer['name']}_{depth['label']}"] = None if val is None else val / d_factor

    zone_soil = pd.DataFrame([row])          # one row, one column per property and depth
    zone_soil.to_csv(OUT / f"{zone}_soilgrids_static.csv", index=False)
    print("saved", zone)

saved ecija
saved ubeda
saved guadix
saved jerez


### 3. About the SRTM data 
**What it is.** SRTM (Shuttle Radar Topography Mission) is a NASA elevation dataset
collected in February 2000 with radar. We use the 1 arc-second product
(`SRTMGL1`, about 30 m), downloaded through AppEEARS as a small GeoTIFF (about
2 × 2 km) around each zone's reference point. The elevation layer is stored in
meters, and missing pixels are marked -32768.

**Why we use it.** Terrain controls where water runs off and where soil erodes.
Slope computed from the elevation grid separates flat land from sloping land, which
matters most for the olive zone, where bare soil on slopes is the main erosion
risk.

**How we process it.** We mask missing pixels, convert pixel sizes from degrees to
meters (a degree of longitude is shorter than a degree of latitude at this
latitude), [smooth the elevation with a 3 × 3 average], and compute elevation and
slope statistics: mean elevation, mean slope and the 90th percentile of slope. Each
zone gets one row in the static features table.

**Limitations.** SRTM is a single snapshot from 2000. On very flat land its vertical
error adds apparent slope, so slope values are best used to compare zones with each
other and not as exact terrain measurements. The 2 × 2 km box describes the area
around the parcel, not the parcel alone.

### Table 1: Raster quality checks (`all_checks`)

Verifies that each downloaded SRTM file is usable before any statistics are computed.

| Column | What it checks | Result |
|---|---|---|
| `crs` | Coordinate system of the raster (EPSG:4326 = latitude/longitude) | Same in all zones |
| `size` | Raster dimensions in pixels (width, height): about 91 × 73 pixels of 30 m, i.e. a ~2 × 2 km box | Same in all zones |
| `point_inside` | The zone's reference point lies inside the raster | True in all zones, so the files belong to the right locations |
| `share_missing` | Share of pixels with no elevation value (0 = none) | 0.0 in all zones |
| `elev_min_max` | Lowest and highest elevation in the box (m) | Plausible for each zone (Écija 174-190 m, Úbeda 485-635 m, Guadix 1054-1139 m, Jerez 8-128 m) |

In [17]:
rows, all_checks = [], {}
for zone, point in ZONES.items():
    checks, stats = srtm_clean(f"../data/raw/srtm/{zone}_srtm.tif", point)
    all_checks[zone] = checks
    rows.append({"zone": zone, **stats})

pd.DataFrame(all_checks).T  

,crs,size,point_inside,share_missing,elev_min_max
ecija,EPSG:4326,"(91, 73)",True,0.0,"(174.0, 190.0)"
ubeda,EPSG:4326,"(91, 73)",True,0.0,"(485.0, 635.0)"
guadix,EPSG:4326,"(91, 73)",True,0.0,"(1054.0, 1139.0)"
jerez,EPSG:4326,"(91, 73)",True,0.0,"(8.0, 128.0)"


### Table 2: Terrain features per zone (`srtm_static`)

One row per zone. These are the values used in the analysis and in the rotation scoring.

| Column | Meaning | What it tells us |
|---|---|---|
| `zone` | Zone identifier | Key used to join with the other datasets |
| `elev_mean_m` | Mean elevation (m above sea level) | Altitude, which affects temperature and frost risk |
| `elev_std_m` | Standard deviation of elevation within the box (m) | How uneven the terrain is: low = flat, high = rolling or hilly |
| `slope_mean_deg` | Mean slope (degrees), after 3 × 3 smoothing | Typical steepness of the area, an indicator of runoff and erosion exposure |
| `slope_p90_deg` | 90th percentile of slope (degrees) | Steepness of the steepest 10% of the area, showing erosion hotspots |

In [18]:
srtm_static = pd.DataFrame(rows)
srtm_static.round(2)

,zone,elev_mean_m,elev_std_m,slope_mean_deg,slope_p90_deg
0,ecija,181.79,1.87,1.47,2.54
1,ubeda,560.46,31.85,6.23,9.38
2,guadix,1100.69,16.01,3.17,6.69
3,jerez,62.27,25.39,5.10,7.97
